In [2]:
from pyspark.sql import functions as F

bq_project = "examen-final-481401"
bq_dataset = "FinalOrtiz"

# bucket temporal (tiene que existir)
temp_gcs_bucket = "bucket-ortiz-final"  

FACT_TBL = f"{bq_project}.{bq_dataset}.fact_flights_delay"
DIM_CARRIER_TBL = f"{bq_project}.{bq_dataset}.dim_carrier"
DIM_AIRPORT_TBL = f"{bq_project}.{bq_dataset}.dim_airport"

fact = (spark.read.format("bigquery")
        .option("table", FACT_TBL)
        .load())

dim_carrier = (spark.read.format("bigquery")
               .option("table", DIM_CARRIER_TBL)
               .load())

dim_airport = (spark.read.format("bigquery")
               .option("table", DIM_AIRPORT_TBL)
               .load())

print("fact rows:", fact.count())
fact.printSchema()


fact rows: 484551
root
 |-- Date: string (nullable = true)
 |-- UniqueCarrier: string (nullable = true)
 |-- Origin: string (nullable = true)
 |-- Dest: string (nullable = true)
 |-- TailNum: string (nullable = true)
 |-- FlightNum: string (nullable = true)
 |-- DepTime: string (nullable = true)
 |-- ArrTime: string (nullable = true)
 |-- CRSArrTime: string (nullable = true)
 |-- ActualElapsedTime: string (nullable = true)
 |-- CRSElapsedTime: string (nullable = true)
 |-- AirTime: string (nullable = true)
 |-- Distance: string (nullable = true)
 |-- TaxiIn: string (nullable = true)
 |-- TaxiOut: string (nullable = true)
 |-- ArrDelay: string (nullable = true)
 |-- DepDelay: string (nullable = true)
 |-- CarrierDelay: string (nullable = true)
 |-- WeatherDelay: string (nullable = true)
 |-- NASDelay: string (nullable = true)
 |-- SecurityDelay: string (nullable = true)
 |-- LateAircraftDelay: string (nullable = true)
 |-- Cancelled: string (nullable = true)
 |-- CancellationCode: strin

In [3]:
def write_to_bq(df, table_name: str, mode: str = "overwrite"):
    full_table = f"{bq_project}:{bq_dataset}.{table_name}"
    (df.write
       .format("bigquery")
       .option("table", full_table)
       .option("temporaryGcsBucket", temp_gcs_bucket)
       .mode(mode)
       .save()
    )
    print("[OK] escrito:", full_table)



In [4]:
kpi_overview = (fact.agg(
    F.count("*").alias("total_vuelos"),
    F.avg(F.col("ArrDelay").cast("double")).alias("avg_arr_delay"),
    F.avg(F.col("DepDelay").cast("double")).alias("avg_dep_delay"),
    F.avg(F.when(F.col("ArrDelay").cast("double") > 15, 1).otherwise(0)).alias("pct_arr_delay_gt15"),
    F.avg(F.when(F.col("DepDelay").cast("double") > 15, 1).otherwise(0)).alias("pct_dep_delay_gt15"),
    F.avg(F.col("Cancelled").cast("double")).alias("pct_cancelled"),
    F.avg(F.col("Diverted").cast("double")).alias("pct_diverted"),
))

kpi_overview.show(truncate=False)
write_to_bq(kpi_overview, "oro_kpi_overview")


25/12/16 05:56:00 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+------------+-----------------+-----------------+------------------+------------------+-------------+------------+
|total_vuelos|avg_arr_delay    |avg_dep_delay    |pct_arr_delay_gt15|pct_dep_delay_gt15|pct_cancelled|pct_diverted|
+------------+-----------------+-----------------+------------------+------------------+-------------+------------+
|484551      |60.90776409500754|57.49808585680351|0.9731276996642252|0.8788693037471804|0.0          |0.0         |
+------------+-----------------+-----------------+------------------+------------------+-------------+------------+



[OK] escrito: examen-final-481401:FinalOrtiz.oro_kpi_overview


In [6]:
kpi_by_carrier = (fact
    .join(dim_carrier.select("UniqueCarrier", "Airline"),
          on="UniqueCarrier", how="left")
    .groupBy("UniqueCarrier", "Airline")
    .agg(
        F.count("*").alias("vuelos"),
        F.avg(F.col("ArrDelay").cast("double")).alias("avg_arr_delay"),
        F.avg(F.col("DepDelay").cast("double")).alias("avg_dep_delay"),
        F.expr("percentile_approx(cast(ArrDelay as double), 0.95)").alias("p95_arr_delay"),
        F.avg(F.when(F.col("ArrDelay").cast("double") > 15, 1).otherwise(0)).alias("pct_arr_delay_gt15"),
        F.avg(F.col("Cancelled").cast("double")).alias("pct_cancelled"),
    )
    .orderBy(F.col("vuelos").desc())
)

kpi_by_carrier.show(10, truncate=False)
write_to_bq(kpi_by_carrier, "oro_kpi_by_carrier")


+-------------+----------------------------+------+------------------+------------------+-------------+------------------+-------------+
|UniqueCarrier|Airline                     |vuelos|avg_arr_delay     |avg_dep_delay     |p95_arr_delay|pct_arr_delay_gt15|pct_cancelled|
+-------------+----------------------------+------+------------------+------------------+-------------+------------------+-------------+
|WN           |Southwest Airlines Co.      |119048|51.032944694576976|51.24512801559035 |142.0        |0.9648125125999597|0.0          |
|AA           |American Airlines Inc.      |73053 |65.72962096012485 |60.31490835420859 |175.0        |0.9798502457120173|0.0          |
|MQ           |American Eagle Airlines Inc.|58698 |64.27723261439913 |57.815240723704385|166.0        |0.9770690653855327|0.0          |
|UA           |United Air Lines Inc.       |56896 |69.67053922947132 |66.36900660854893 |189.0        |0.9779949381327334|0.0          |
|OO           |Skywest Airlines Inc.     

[OK] escrito: examen-final-481401:FinalOrtiz.oro_kpi_by_carrier


In [7]:
kpi_trend_daily = (fact
    .filter(F.col("flight_date").isNotNull())
    .groupBy("flight_date")
    .agg(
        F.count("*").alias("vuelos"),
        F.avg(F.col("ArrDelay").cast("double")).alias("avg_arr_delay"),
        F.avg(F.col("DepDelay").cast("double")).alias("avg_dep_delay"),
        F.avg(F.when(F.col("ArrDelay").cast("double") > 15, 1).otherwise(0)).alias("pct_arr_delay_gt15"),
    )
    .orderBy("flight_date")
)

kpi_trend_daily.show(10, truncate=False)
write_to_bq(kpi_trend_daily, "oro_kpi_trend_daily")


+-----------+------+------------------+------------------+------------------+
|flight_date|vuelos|avg_arr_delay     |avg_dep_delay     |pct_arr_delay_gt15|
+-----------+------+------------------+------------------+------------------+
|2019-01-01 |3759  |64.01755786113328 |60.280925778132485|0.9683426443202979|
|2019-01-02 |4958  |70.84731746672045 |67.72650262202501 |0.9802339653085922|
|2019-01-03 |4621  |53.56676044146289 |52.02531919497944 |0.9679723003678857|
|2019-01-04 |3472  |59.01123271889401 |54.42425115207373 |0.9740783410138248|
|2019-01-05 |2053  |54.80077934729664 |49.405747686312715|0.9683390160740379|
|2019-01-06 |1701  |50.001763668430335|48.65961199294533 |0.9594356261022927|
|2019-02-01 |5256  |53.01731354642313 |53.226027397260275|0.9748858447488584|
|2019-02-02 |1845  |58.0319783197832  |55.43577235772358 |0.9696476964769648|
|2019-02-03 |2798  |56.12223016440314 |51.323087919942814|0.9721229449606862|
|2019-02-04 |1802  |47.08046614872364 |45.8873473917869  |0.9600

[OK] escrito: examen-final-481401:FinalOrtiz.oro_kpi_trend_daily


In [9]:
# KPI: Por ruta (Origen - Destino)
kpi_by_route = (fact
    .groupBy("Origin", "Dest")
    .agg(
        F.count("*").alias("vuelos"),
        F.avg(F.col("ArrDelay").cast("double")).alias("avg_arr_delay"),
        F.avg(F.when(F.col("ArrDelay").cast("double") > 15, 1).otherwise(0)).alias("pct_arr_delay_gt15"),
        F.avg(F.col("Cancelled").cast("double")).alias("pct_cancelled"),
    )
    .filter(F.col("vuelos") >= 20)
    .orderBy(F.col("avg_arr_delay").desc(), F.col("vuelos").desc())
    .limit(200)
)

# Guardar el KPI de rutas en BigQuery (tabla ORO)
write_to_bq(kpi_by_route, "oro_kpi_by_route")



[OK] escrito: examen-final-481401:FinalOrtiz.oro_kpi_by_route


In [12]:
from pyspark.sql import functions as F

kpi_by_route = (fact
    .groupBy("Origin", "Dest")
    .agg(
        F.count("*").alias("vuelos"),
        F.avg(F.col("ArrDelay").cast("double")).alias("avg_arr_delay"),
        F.avg(F.when(F.col("ArrDelay").cast("double") > 15, 1).otherwise(0)).alias("pct_arr_delay_gt15"),
        F.avg(F.col("Cancelled").cast("double")).alias("pct_cancelled"),
    )
    .filter(F.col("vuelos") >= 20)
    .orderBy(F.col("avg_arr_delay").desc(), F.col("vuelos").desc())
    .limit(200)
)

kpi_by_route.show(10, truncate=False)
write_to_bq(kpi_by_route, "oro_kpi_by_route")



+------+----+------+------------------+------------------+-------------+
|Origin|Dest|vuelos|avg_arr_delay     |pct_arr_delay_gt15|pct_cancelled|
+------+----+------+------------------+------------------+-------------+
|EGE   |LAX |32    |148.4375          |1.0               |0.0          |
|GUC   |DFW |29    |141.44827586206895|1.0               |0.0          |
|HNL   |PHX |78    |137.43589743589743|0.9743589743589743|0.0          |
|MQT   |MKE |33    |133.03030303030303|1.0               |0.0          |
|SAN   |HNL |20    |132.55            |0.95              |0.0          |
|LAS   |HNL |33    |131.4848484848485 |1.0               |0.0          |
|EGE   |DFW |71    |128.56338028169014|0.9859154929577465|0.0          |
|RSW   |STL |25    |128.04            |0.96              |0.0          |
|HDN   |ORD |81    |125.77777777777777|1.0               |0.0          |
|MKE   |MQT |75    |124.66666666666667|0.9866666666666667|0.0          |
+------+----+------+------------------+------------

[OK] escrito: examen-final-481401:FinalOrtiz.oro_kpi_by_route
